# Lecture: Linear function approximation for the Mountain Car environment

## Introduction to the Mountain Car Environment

The Mountain Car environment is a classic benchmark problem in reinforcement learning, provided by Gymnasium. It simulates a car stuck in a valley between two hills, where the goal is to drive up the right hill and reach the flag at the top.

However, the car's engine is not powerful enough to drive directly up the hill. Instead, the agent must learn to build momentum by first driving back and forth between the hills to reach the goal.

<img src="images/mountain_car.gif" alt="Mountain Car" width="400"/>

### Environment Details

- **State space**: A 2D continuous space:
  - `position` ∈ [-1.2, 0.6]
  - `velocity` ∈ [-0.07, 0.07]

- **Action space**: Discrete with 3 actions:
  - `0`: Push left
  - `1`: Do nothing
  - `2`: Push right

- **Reward**: -1 at each time step until the goal is reached.

- **Objective**: Reach the goal (position ≥ 0.5) in as few steps as possible.

Run the following cell only if you are working with google colab to copy the required .py file in the root directory. If you are working locally just ignore this cell!

In [ ]:
!git clone https://github.com/Fjoelsak/RL.git
!cp RL/06_Value_Function_Approximation/LinearSarsaAgent.py RL/06_Value_Function_Approximation/TileCoding.py ./
!mkdir images
!cp RL/06_Value_Function_Approximation/images/mountain_car.gif ./images

## Linear function approximation with SARSA

Understand the code in 'LinearSarsaAgent.py' and 'TileCoding.py' and try to figure out the impact of the tilings and bins as well as alpha on the training performance. Vary the number of episodes to train and use the 'plot_learning_curve()' method for an insight in the training process

### What the three hyperparameters control

The continuous Mountain Car state has to be turned into discrete features before a linear model can learn from it. Tile coding does this with several overlapping grids. Three knobs govern that representation and the learning speed:

- **`bins`** — the number of tiles *per dimension within a single grid*. `bins=(8, 8)` splits both `position` and `velocity` into 8 strips, giving an 8×8 = 64-tile grid. **Higher bins → finer resolution** (the agent can tell nearby states apart), but more weights to learn and weaker generalization, so training needs more episodes.

- **`tilings`** — the number of *overlapping grids*, each shifted by a small offset. With `tilings=10`, every state activates exactly 10 tiles (one per grid). **More tilings → smoother generalization** and a more expressive value function, at higher computational cost. This is the main lever for sample efficiency.

- **`alpha`** — the base learning rate. Internally it is divided by `tilings` (`self.alpha = alpha / tilings`), because all active tiles contribute to one Q-value, so their combined update must match the intended step size. **Too high → unstable / diverging** returns; **too low → very slow** learning. With tile coding values around `0.1`–`0.5` are typical.

### How to read the results

Mountain Car gives **-1 per step** and truncates at 200 steps, so an untrained agent sits at **-200** and a better policy moves the curve *upward* (fewer steps to the flag). Use `plot_learning_curve()`: the moving average reveals the trend, while raw per-episode returns show the variance. When sweeping a parameter, change **one knob at a time** and keep the episode count fixed to make runs comparable.

In [ ]:
import gymnasium as gym
from LinearSarsaAgent import LinearSarsaAgent

# Training Loop
env = gym.make('MountainCar-v0')

# Hyperparameters of the tile-coded SARSA agent (vary these to study their effect):
#   tilings=10   -> 10 overlapping, offset grids; main lever for generalization
#   bins=(8, 8)  -> 8x8 tiles per grid; resolution along position and velocity
#   alpha=.1     -> base learning rate (divided by tilings internally)
agent = LinearSarsaAgent(env, tilings=10, bins=(8, 8), alpha=.1)

# Train for a fixed number of episodes; keep this constant when comparing runs.
returns = agent.train(1500)
env.close()

### Learning curve

Plot the per-episode return to judge whether training converged. The red moving average exposes the trend, while the faint raw curve shows the episode-to-episode variance. A curve rising from -200 toward less negative values means the agent reaches the flag in fewer steps. If it stays flat at -200, try more episodes, a higher `alpha`, or more `tilings`.

In [ ]:
agent.plot_learning_curve(returns)

### Visualizing the learned value function (cost-to-go) and the agent's path

Beyond the learning curve, we can *look inside* the agent and see what it actually learned. Tile coding gives us an action-value function $Q(s, a)$ over the continuous state $s = (\text{position}, \text{velocity})$. A common way to visualize it is the **cost-to-go function**

$$ J(s) = -\max_a Q(s, a), $$

i.e. the (negated) value of the best action in each state. Since every step costs $-1$, $\max_a Q(s,a)$ is roughly *minus the number of steps still needed to reach the flag*, so the cost-to-go $J(s)$ is approximately **how many steps the agent expects to still need** from that state.

The cell below sweeps a fine grid over the whole state space, queries the trained agent at each point, and draws $J(s)$ as a surface — the classic Mountain Car plot from Sutton & Barto. On top of the surface we **overlay the actual trajectory** the greedy agent takes in one episode. Starting at rest near `position≈-0.5`, the car cannot climb the hill directly, so it **swings back and forth several times**, adding a bit of energy with each swing until it has enough momentum to reach the flag. In the (position, velocity) plane these growing swings trace a **spiral** — that is exactly what the red path shows. **You don't need to implement anything; just run it and follow the red line from the green start marker to the red goal marker.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Helper: cost-to-go J(s) = -max_a Q(s, a) for a single state.
def cost_to_go_at(state):
    return -np.max([agent.get_q(state, a) for a in range(agent.n_actions)])

# 1) Sweep a fine grid over the full continuous state space.
low, high = env.observation_space.low, env.observation_space.high
positions = np.linspace(low[0], high[0], 100)
velocities = np.linspace(low[1], high[1], 100)
P, V = np.meshgrid(positions, velocities)

cost_to_go = np.zeros_like(P)
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        cost_to_go[i, j] = cost_to_go_at((P[i, j], V[i, j]))

# 2) Roll out one greedy episode and record the visited states.
eval_env = gym.make('MountainCar-v0')
state, _ = eval_env.reset()
traj_pos, traj_vel, traj_cost = [], [], []
done = False
while not done:
    traj_pos.append(state[0])
    traj_vel.append(state[1])
    traj_cost.append(cost_to_go_at(state))
    action = int(np.argmax([agent.get_q(state, a) for a in range(agent.n_actions)]))
    state, _, terminated, truncated, _ = eval_env.step(action)
    done = terminated or truncated
eval_env.close()

# 3) Surface + trajectory overlay.
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(P, V, cost_to_go, cmap='viridis', edgecolor='none', alpha=0.6)

# The path the greedy agent actually drives through state space.
ax.plot(traj_pos, traj_vel, traj_cost, color='red', linewidth=2, label='Greedy trajectory')
ax.scatter(traj_pos[0], traj_vel[0], traj_cost[0],
           color='lime', s=60, label='Start')
ax.scatter(traj_pos[-1], traj_vel[-1], traj_cost[-1],
           color='red', s=60, marker='*', label='Goal reached')

ax.set_xlabel('Position')
ax.set_ylabel('Velocity')
ax.set_zlabel('Cost-to-go  -max_a Q(s, a)')
ax.set_title('Learned cost-to-go function with the agent\'s greedy path')
ax.view_init(elev=40, azim=-120)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Episode length (steps to the flag): {len(traj_pos)}")

### How to read this plot

The height of the surface is the **expected number of steps still needed** to reach the flag from each state (higher = worse). The red line is the path the greedy agent actually drives, from the green start to the red goal star.

- **The red path spirals because the car swings back and forth several times.** Each swing is one loop in the (position, velocity) plane: roll left (velocity goes negative), get braked by the left slope (velocity back to positive), accelerate right (position rises but not far enough), get braked by the right slope (velocity negative again) — and repeat. Because the agent adds a little energy on every swing, each loop is *larger* than the last, so the loops do not close into a circle but **wind outward into a spiral**. The spiral therefore shows *how many oscillations* the agent needs to build enough momentum.

- **Why "away from the goal" can be good.** Because the engine is too weak to climb directly, states with the right *velocity* matter more than states close to the goal in *position*. The first thing the agent does is drive **left, away from the flag**, to store potential energy. The value surface encodes exactly this: some far-left states have a lower cost-to-go than states that are closer to the flag but have the wrong velocity.

- **Ridges and plateaus** mark states from which the car cannot reach the goal directly and must oscillate first, costing many steps.

- **The blocky, faceted look** is tile coding showing through: the value function is piecewise constant within tiles and only smoothed by the overlap of the `tilings` offset grids. **Re-run the training with fewer `bins` or fewer `tilings`** and the surface becomes coarser; with more, it becomes smoother — a direct, *visual* version of the hyperparameter study above.

- **If the path times out** (length close to 200 and never reaching `position≥0.5`), the agent has not learned a working policy yet; train for more episodes or raise `alpha`.

### Visualizing the learned policy locally

Run the trained agent for one episode with a live render window (`render_mode='human'`). This opens a separate window showing the car driving, so it only works in a **local** environment with a display — not in Google Colab. For Colab, use the video-recording cells below instead.

In [ ]:
env = gym.make('MountainCar-v0', render_mode='human')
agent.run_episode(env, render = True)
env.close()

### Note: Use the following code if you are using google colab

In [ ]:
import gymnasium as gym
import numpy as np
from IPython.display import HTML
from base64 import b64encode

import imageio

# instantiation of the environment
env = gym.make("MountainCar-v0", render_mode="rgb_array")

state = env.reset()[0]

# initialize a list of frames for video creation
frames = []

done = False
while not done:
    # capture the frame and append it to frames list
    frame = env.render()
    frames.append(frame)

    # greedy action selection using learned weights
    q_values = [agent.get_q(state, a) for a in range(agent.n_actions)]
    action = int(np.argmax(q_values))

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    # final rendering for last frame of episode
    if done:
        frame = env.render()
        frames.append(frame)

env.close()

# save video
video_path = "./mountain_car.mp4"
imageio.mimsave(video_path, frames, fps=30)

In [ ]:
# this is for displaying the video after saving
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")